# 量子ビットと共振器

このノートブックでは、Qubex のシミュレータ機能を使って、量子ビットと共振器が結合した系を調べます。
量子ビットと共振器の結合系は、回路 QED における最も基本的なモデルの 1 つであり、励起の交換、分散シフト、dressed state など多くの現象の出発点になります。

## 背景

量子ビットを二準位系、共振器を調和振動子として近似すると、結合系は Jaynes-Cummings モデルで表せます。
回転波近似のもとでハミルトニアンは

$$
H = \omega_q \sigma_+ \sigma_- + \omega_r a^\dagger a + g (a^\dagger \sigma_- + a \sigma_+)
$$

となります。

- $\omega_q$: 量子ビットの遷移角周波数
- $\omega_r$: 共振器の角周波数
- $g$: 結合強度
- $a, a^\dagger$: 共振器の消滅・生成演算子
- $\sigma_-, \sigma_+$: 量子ビットの下降・上昇演算子

このハミルトニアンでは、全励起数

$$
N = a^\dagger a + \sigma_+\sigma_-
$$

が保存されるため、各励起数の部分空間ごとに独立に考えることができます。
特に 1 励起部分空間 $\{\ket{g,1}, \ket{e,0}\}$ では、ハミルトニアンは

$$
H_{N=1} =
\begin{pmatrix}
\omega_r & g \\
g & \omega_q
\end{pmatrix}
$$

と書けます。
ここで detuning を $\Delta = \omega_r - \omega_q$ とすると、固有状態は bare state $\ket{g,1}$ と $\ket{e,0}$ の重ね合わせになり、系の固有モードは dressed state になります。

共鳴条件 $\Delta = 0$ では、1 励起は量子ビットと共振器の間をコヒーレントに交換します。
一方、$|\Delta| \gg g$ の分散領域では直接の励起交換は小さいものの、共振器側から量子ビット様モードを弱く駆動できます。

### 共振器側からの量子ビット様モードの駆動

共振器に量子ビットの周波数でマイクロ波をドライブするとします。

$$
H_{d}(t) = A \cos (\omega_q t) (a + a^\dagger)
$$

系全体のハミルトニアンは次のように書けます。

$$
H(t) = H_0 + V + H_d(t)
$$

ここで、

$$
\begin{align*}
H_0 &= \omega_q \sigma_+ \sigma_- + \omega_r a^\dagger a \\
V   &= g (a^\dagger \sigma_- + a \sigma_+) \\
H_{d}(t) &= A \cos (\omega_q t) (a + a^\dagger)
\end{align*}
$$

とします。

$U_d(t) = \exp(-i \omega_q t (a^\dagger a + \sigma_+ \sigma_-))$ で回転波近似を行った $H_{\text{rot}}(t) = U_d^\dagger H U_d - i U_d^\dagger U_d$ は次のようになります。

$$
H_{\text{rot}} = \Delta a^\dagger a + g (a^\dagger \sigma_- + a \sigma_+) + \frac{A}{2} (a + a^\dagger)
$$

次にで Schrieffer-Wolff 変換を行い非対角成分 $V$ を削除すると次のようになります。

$$
H_{\text{eff}} \approx \Delta a^\dagger a + \chi a^\dagger a \sigma_z + \frac{\chi}{2} \sigma_z + \frac{A}{2} (a + a^\dagger) - \frac{Ag}{2\Delta} \sigma_x
$$

$\chi = \frac{g^2}{\Delta}$ です。

<details>
<summary>導出</summary>

$[\Delta a^\dagger a, S] = -V$ となる $S$ を選びます。今回は $S = \lambda(a \sigma_+ - a^\dagger \sigma_-)$ とします。

$$
\begin{align*}
[a^\dagger a, a\sigma_+ - a^\dagger \sigma_-] 
&= [a^\dagger a, a] \sigma_+ - [a^\dagger a, a^\dagger] \sigma_- \\
&= -a \sigma_+ - a^\dagger \sigma_-
\end{align*}
$$

よって $\lambda = \frac{g}{\Delta}$ とすると等しくなります。
この $S$ のもとで、ハミルトニアンを二次まで求めます。 $H_{\text{rot}} = H_{\text{diag}} + V + H_d$ として、

$$
H^\prime = H_{\text{diag}} + H_d + [H_d, S] + \frac{1}{2}[V, S]
$$

のようになります。
いろいろ計算すると、

$$
\begin{align*}
\frac{1}{2} [V, S] &\sim \chi a^\dagger a \sigma_z + \frac{\chi}{2} \sigma_z \\
[H_d, S] &= -\frac{Ag}{2\Delta} \sigma_x
\end{align*}
$$

となります。
$\chi = \frac{g^2}{\Delta}$ です。
以上より、分散領域での有効ハミルトニアは次のようになります。

$$
H_{\text{eff}} \approx \Delta a^\dagger a + \chi a^\dagger a \sigma_z + \frac{\chi}{2} \sigma_z + \frac{A}{2} (a + a^\dagger) - \frac{Ag}{2\Delta} \sigma_x
$$

</details>

共振器の微小変位を無視して量子ビットだけに注目すると $H_q^{\text{eff}} \approx \frac{\chi}{2} \sigma_z - \frac{\Omega_R}{2} \sigma_x$ です。
ただし $\Omega_R = \frac{Ag}{\Delta}$ としています。
量子ビットはラビ振動することがわかります。

このノートブックでは以下のように値を設定しているため、ラビ周波数はおよそ 2.02 MHz となります。

- $\frac{g}{2\pi} = 0.01 \text{GHz}$
- $\frac{\Delta}{2\pi} = \omega_r - \omega_q = 2.475 \text{GHz}$
- $A = 2\pi \times 0.5$
- $\Omega_R = \frac{Ag}{\Delta} = 0.00202 \text{GHz} \times 2\pi$

1000 ns のシミュレーションを行うと、およそ 2 周することがわかります。

In [ ]:
import numpy as np
import qutip as qt

from qubex.simulator import (
    Control,
    Coupling,
    QuantumSimulator,
    QuantumSystem,
    Qubit,
    Resonator,
)

量子ビット `Q01` と共振器 `R01` を用意します。
Qubex の simulator では、これらを `QuantumSystem` にまとめ、さらに `Coupling` で相互作用を指定します。

今回は

- 量子ビット周波数: $\omega_q / 2\pi = 7.648\,\mathrm{GHz}$
- 共振器周波数: $\omega_r / 2\pi = 10.123\,\mathrm{GHz}$
- 結合強度: $g / 2\pi = 0.01\,\mathrm{GHz}$

とします。
したがって detuning は

$$
\Delta / 2\pi = 10.123 - 7.648 = 2.475\,\mathrm{GHz}
$$

であり、$|\Delta| \gg g$ が成り立つ分散的な設定です。
また、共振器は `dimension=10` として 10 準位まで切り詰めています。


In [2]:
# Create the quantum system with a qubit and a resonator (Jaynes-Cummings model)

qubit = Qubit(
    label="Q01",
    frequency=7.648,
)

resonator = Resonator(
    label="R01",
    dimension=10,
    frequency=10.123,
)

coupling_strength = 0.01
detuning = resonator.frequency - qubit.frequency

system = QuantumSystem(
    objects=[qubit, resonator],
    couplings=[
        Coupling(
            pair=(qubit, resonator),
            strength=coupling_strength,
        ),
    ],
)

simulator = QuantumSimulator(system)

print(f"detuning: {detuning:.3f} GHz")
print(f"g / detuning: {coupling_strength / detuning:.4f}")


detuning: 2.475 GHz
g / detuning: 0.0040


系全体のハミルトニアンを確認します。
`Qubit` は 2 準位、`Resonator` は 10 準位なので、全体のヒルベルト空間の次元は $2 \times 10 = 20$ です。
そのため、`system.hamiltonian` は $20 \times 20$ 行列として表示されます。

基底は概念的には

$$
\{\ket{g,0}, \ket{g,1}, \ldots, \ket{g,9}, \ket{e,0}, \ket{e,1}, \ldots, \ket{e,9}\}
$$

の順に並んでいます。
対角成分は bare な量子ビットと共振器のエネルギーであり、非対角成分は結合項

$$
g (a^\dagger \sigma_- + a \sigma_+)
$$

から来ています。
この結合項により、例えば $\ket{g,n+1}$ と $\ket{e,n}$ の間に大きさ $g\sqrt{n+1}$ の結合が入ります。
表示される行列の非対角成分が、その励起交換の強さを表しています。


In [13]:
np.set_printoptions(precision=3, suppress=True, linewidth=300)

np.real(system.hamiltonian.full())

array([[  0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ],
       [  0.   ,  63.605,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.063,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ],
       [  0.   ,   0.   , 127.209,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.089,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ],
       [  0.   ,   0.   ,   0.   , 190.814,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.109,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ],
       [  0.   ,   0.   ,   0.   ,   0.   , 254.419,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.126,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ],
       [  0.   ,   0.   ,   0.   ,   0.   ,   0.   , 31

次に、共振器チャネルに drive を与えます。
drive 周波数は量子ビット周波数 `qubit.frequency` に設定します。

これは分散領域での dressed mode を意識した設定です。
bare 共振器そのものは 10.123 GHz にありますが、結合系の固有モードのうち下側のモードは量子ビットに強く、共振器演算子にもわずかな重みを持ちます。
そのため、共振器ポートからでも量子ビット様モードを弱く駆動できます。

時間依存の drive 項は概念的には

$$
H_d(t) = \epsilon(t) (a + a^\dagger)
$$

の形で加わります。
ここでは振幅一定の長い矩形パルスを使い、狭い周波数帯域で量子ビット様モードを選択的に励起します。


In [4]:
control = Control(
    target=resonator.label,
    frequency=qubit.frequency,
    waveform=[2 * np.pi * 0.5] * 1000,
    durations=[1.0] * 1000,
)
control.plot()


初期状態は量子ビット・共振器ともに基底状態、すなわち $\ket{g,0}$ とします。
そのうえで `QuantumSimulator.mesolve` を使って時間発展を計算します。

`mesolve` は一般には Lindblad 方程式

$$
\frac{d\rho}{dt} = -i[H(t), \rho] + \sum_k \left( L_k \rho L_k^\dagger - \frac{1}{2}\{L_k^\dagger L_k, \rho\} \right)
$$

に基づく時間発展を解きます。
今回は損失や緩和を明示していないため、主にハミルトニアンに由来するコヒーレントなダイナミクスを見ることになります。


In [5]:
result = simulator.mesolve(
    controls=[control],
    initial_state={
        "Q01": qt.basis(2, 0),
        "R01": qt.basis(10, 0),
    },
)

結果を可視化します。

- `plot_population_dynamics(qubit.label)`: 量子ビットの基底・励起状態の占有率
- `plot_population_dynamics(resonator.label)`: 共振器のフォック状態 $\ket{n}$ の占有率
- `display_bloch_sphere(qubit.label)`: 量子ビットの Bloch 球上での運動

この例では共振器ポートから量子ビット様モードを駆動しているため、共振器に大きくエネルギーを蓄えるというより、dressed state を介して量子ビット側に応答が現れます。
もし $\omega_q$ と $\omega_r$ を近づければ、励起交換はより強くなり、量子ビットと共振器の間を行き来する真空ラビ振動に近い振る舞いが見えやすくなります。


In [6]:
result.plot_population_dynamics(qubit.label)
result.plot_population_dynamics(resonator.label)
result.display_bloch_sphere(qubit.label)


<IPython.core.display.Javascript object>

## まとめ

このノートブックでは、量子ビットと共振器からなる Jaynes-Cummings 系を Qubex の `QuantumSimulator` で記述し、
共振器ポートからの drive に対する応答を確認しました。

重要な点は次の 3 つです。

- 結合系では bare な量子ビット・共振器状態ではなく dressed state が自然な固有状態になること
- 分散領域では励起交換は弱いが、共振器演算子を通して量子ビット様モードを間接的に駆動できること
- `QuantumSystem` と `Coupling` を使うことで、こうした結合系のハミルトニアンと時間発展をそのまま数値的に調べられること

次の発展としては、共鳴条件に近いパラメータに変更して真空ラビ振動を観察したり、損失を導入して開放系としてのダイナミクスを比較したりすると、量子ビット-共振器系の理解が深まります。
